In [1]:
# 필요하면 SASRec distillation 추가
"""
E5 (User's LoRA): 이미 책 내용을 잘 이해하도록 튜닝해두셨죠? 이건 가중치를 고정합니다. (업데이트 X)
Add-on Layer: 카테고리 임베딩 레이어(Category Embedding)와 이 둘을 합치는 퓨전 레이어(Linear Layer)만 새로 만듭니다.
학습: 데이터 흘려보내면서 새로 만든 작은 레이어들만 학습시킵니다.
모델 구조를 뜯어고치는 건 나중에 '평점'이나 '페이지 수' 같은 숫자를 꼭 넣어야 할 때 하셔도 늦지 않습니다.
"""
import random
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType #, AdapterConfig

In [2]:
# !pip install -U transformers

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능
    
set_seed(42)

In [4]:
# book_path = './data/e5_book_meta.parquet'
book_path = './data/book_for_emb.parquet'
books = pd.read_parquet(book_path)

In [5]:
# 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
def build_text(row):
    parts = [
        f"Title: {row['title']} |",
        # f"Category: {row['category']} |",
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Category: ... Description: ..."

books["text"] = books.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

In [6]:
books["text"].iloc[1220]

'Title: Someone Knows My Name | Description: Abducted from Africa as a child and enslaved in South Carolina, Aminata Diallo thinks only of freedom--and of the knowledge, she needs to get home. Sold to an indigo trader who recognizes her intelligence, Aminata is torn from her husband and child and thrown into the chaos of the Revolutionary War. In Manhattan, Aminata helps pen the Book of Negroes, a list of blacks rewarded for service to the king with safe passage to Nova Scotia. There Aminata finds a life of hardship and stinging prejudice. When the British abolitionists come looking for "adventurers" to create a new colony in Sierra Leone, Aminata assists in moving 1,200 Nova Scotians to Africa and aiding the abolitionist cause by revealing the realities of slavery to the British public.\nThis captivating story of one woman\'s remarkable experience spans six decades and three continents and brings to life a crucial chapter in world history.'

In [7]:
# 100개 미만인 카테고리는 노이즈로 간주하고 삭제 추천
counts = books['category'].value_counts()
valid_categories = counts[counts > 100].index
books = books[books['category'].isin(valid_categories)]

In [8]:
dataset = Dataset.from_pandas(books)

le = LabelEncoder()
le.fit(dataset['category'])   # 전체 데이터로 학습

def encode_label(x):
    return {"label": le.transform([x["category"]])[0]}

dataset = dataset.map(encode_label)

num_classes = len(le.classes_)

Map:   0%|          | 0/81845 [00:00<?, ? examples/s]

In [9]:
# model_name = "intfloat/e5-small" 
model_name = "intfloat/e5-small" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32, # 8,
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, lora_config)

In [10]:
# """
# LoRA: Multi-Head Attention 내부의 Wq, Wk, Wv, Wo 일부만 학습
# → 문맥 맞추는 능력 강화

# Adapter: FFN 층 사이에 bottleneck layer 삽입
# → 추가적인 “저장 공간” 확보 (도메인 지식 저장)
# """
# from peft import IA3Config

# ia3_config = IA3Config(
#     task_type=TaskType.CAUSAL_LM,
#     feedforward_modules=["mlp"],
#     inference_mode=False
# )

In [11]:
def collate_fn(batch):
    # texts = [f"passage: {x['text']}" for x in batch]
    texts = [f"query: {x['text']}" for x in batch]
    labels = torch.tensor([x['label'] for x in batch])

    inputs = tokenizer(
      texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    
    return inputs, labels

In [12]:
total_len = len(dataset)
train_len = int(total_len * 0.8)
valid_len = total_len - train_len

train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn
)

In [13]:
def compute_retrieval_accuracy(model, dataloader, device, k=10):
    embeddings_list = []
    labels_list = []

    with torch.no_grad():
        for batch_inputs, labels in dataloader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            labels = labels.to(device)

            outputs = model(**batch_inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = F.normalize(embeddings, p=2, dim=1)

            embeddings_list.append(embeddings)
            labels_list.append(labels)

    all_embeddings = torch.cat(embeddings_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)
    similarity = torch.matmul(all_embeddings, all_embeddings.T)
    similarity.fill_diagonal_(-1e9)

    # nn_index = similarity.argmax(dim=1) # 가장 유사한 이웃 인덱스
    # nn_labels = all_labels[nn_index] # nearest neighbor label
    # accuracy = (nn_labels == all_labels).float().mean().item()

    topk_vals, topk_idx = similarity.topk(k, dim=1) # top-k neighbor 인덱스
    nn_labels_topk = all_labels[topk_idx]  # (N, k) 
    # correct_ratio = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1)
    # accuracy = correct_ratio.mean().item()
    # return accuracy

    # -----------------------------
    # top-1: 가장 가까운 neighbor 정답 여부
    # top-k: k개 안에 정답이 하나라도 있으면
    # precision@k: k개 neighbor 중 정답 비율 평균
    # MRR: 정답이 rank 몇 번째인지에 따른 평균 역수 → rank 1이면 1.0, rank 2이면 0.5
    # -----------------------------
    # Top-1 accuracy
    top1_labels = nn_labels_topk[:, 0]
    top1_acc = (top1_labels == all_labels).float().mean().item()
    
    # Top-k accuracy (at least 1 match)
    topk_match = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1) # .any(dim=1)
    topk_acc = topk_match.float().mean().item()

    # Precision@k
    precision_at_k = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1).mean().item()
    
    # MRR (Mean Reciprocal Rank)
    ranks = (nn_labels_topk == all_labels.unsqueeze(1)).float() # 정답 label 위치 찾기
    reciprocal_rank = []
    for i in range(ranks.size(0)):
        pos_positions = torch.nonzero(ranks[i]).flatten()
        if len(pos_positions) == 0: # positive 없으면 reciprocal rank = 0
            reciprocal_rank.append(0.0)
        else:
            reciprocal_rank.append(1.0 / (pos_positions[0].item() + 1))
    mrr = sum(reciprocal_rank) / len(reciprocal_rank)

    metrics = {
        "top1_acc": top1_acc,
        "topk_acc": topk_acc,
        "precision@k": precision_at_k,
        "mrr": mrr
    }
    return metrics

In [14]:
# 일단 한번 해보고 개선되면 memory bank 식으로 바꿔보자
# 근데 배치가 크면 굳이 memory bank로 안바꿔도 된다는디?
def hard_negative(embeddings, labels, k=3):
    batch_size = embeddings.size(0)
    similarity = torch.matmul(embeddings, embeddings.T) # 1. 유사도 행렬 계산
    
    # hard_neg_sims = []
    # for i in range(batch_size): # for문 대신 행렬 연산(Vectorization)으로 바꾸면 빨라진대
    #     mask = labels != labels[i]           # 다른 라벨만
    #     sim_row = similarity[i].clone()
    #     sim_row[~mask] = -1e9                # 같은 라벨 제외
    #     topk_sim, _ = sim_row.topk(k)        # 가장 가까운 k개
    #     hard_neg_sims.append(topk_sim)
    # hard_neg_sims = torch.stack(hard_neg_sims, dim=0)  # (batch_size, k)

    # 2. Positive Mask 생성 (같은 라벨인 곳 True)
    # labels: (N) -> (N, 1) == (1, N) 브로드캐스팅
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)
    
    # 3. 같은 라벨(Positive + 자기자신)은 유사도를 -무한대로 보내서 topk에서 제외
    # clone()을 안 쓰면 원본 similarity가 망가지므로 주의 (필요시)
    similarity.masked_fill_(pos_mask, -1e9) 

    # 4. 각 행(Query)마다 가장 높은 유사도를 가진 k개(Hard Negatives) 추출
    hard_neg_sims, _ = similarity.topk(k, dim=1) # (N, k)

    return hard_neg_sims

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

EPOCHS = 20
WARMUP_RATIO = 0.1
LEARNING_RATE = 2e-4 # [new] v1:3e-5 / v2:1e-4 / v3:3e-4 / *v4:6e-4*
# [warmup=0.1] v1:3e-4 / v2:3e-5 / v3:1e-4 / v4:2e-4 / v5:4e-4 / *v6:6e-4* / v7:7e-4 
# v8: warmup:0.2, lr:6e-4 / v9: warmup:0.15, lr:6e-4
# ------------- loss 함수 수정 -------------
# v10: warmup:0.1, lr:2e-4

total_steps = len(train_loader) * EPOCHS

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps
)

best_mrr = 0.0
save_path = "./lora_best_retrieval"

In [16]:
import wandb

wandb.init(
    project="retrieval-contrastive", 
    name=f"e5-lora-{LEARNING_RATE}",
    config={
        "lr": LEARNING_RATE,
        "temperature": 0.05,
        "epochs": 20,
        "batch_size": train_loader.batch_size,
    }
)

wandb.watch(model, log="all")  # gradients, parameters 기록

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

In [17]:
metrics = compute_retrieval_accuracy(model, valid_loader, device)
wandb.log({
    "epoch": 0,
    "train_loss": 2,
    "valid/top1_acc": metrics["top1_acc"],
    "valid/top10_acc": metrics["topk_acc"],
    "valid/precision@10": metrics["precision@k"],
    "valid/mrr": metrics["mrr"],
    "lr": LEARNING_RATE,
})
print(f"[Before Training...]")
print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

for epoch in range(20):
    model.train()
    total_train_loss = 0

    for batch_inputs, labels in tqdm(train_loader, desc = f"Epoch: {epoch+1}"):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)
        
        outputs = model(**batch_inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        similarity = torch.matmul(embeddings, embeddings.T)
        
        labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
        identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
        labels_eq = labels_eq & (~identity_mask)

        pos_mask = labels_eq.float()   # (N, N)

        """
        pos_sim = similarity * pos_mask # (N, N) → anchor i의 positive 여러 개 (동일 label 여러 개)
        여기서 pos_mask는 정답(Positive)이 아닌 곳을 0으로 만듭니다. 
        문제는 유사도 점수(Logit)가 0이 된다고 해서, 확률(exp) 계산에서 사라지는 게 아니라는 점입니다.

        "나랑 같은 카테고리가 아닌 놈들(Mask=0)"을 "유사도 0인 상태(exp=1)"로 만들어서 분자와 분모에 다 더해버리고 있습니다. 
        유사도 0은 코사인 유사도에서 '직각'을 의미하며, 이는 꽤 의미 있는 관계일 수 있습니다(완전 반대인 -1보다 훨씬 큼). 
        아예 무시(0으로 취급)하려면 exp 결과가 0이 되어야 합니다.
        """
        # 정답이 아닌 곳(0)에는 아주 작은 값(-1e9)을 넣어서 exp 계산 시 0이 되도록 해야 함
        # pos_sim = similarity.masked_fill(~pos_mask.bool(), -1e9)
        pos_sim = similarity * pos_mask
        neg_sim = hard_negative(embeddings, labels) # (N, k) → anchor i의 negative k개

        temperature = 0.05 # 타우
        pos_sim = pos_sim / temperature
        neg_sim = neg_sim / temperature
        """ 정답 점수: 0.8 / 0.05 = 16
        오답 점수: 0.7 / 0.05 = 14
        exp(16) ≈ 8,886,110 vs exp(14) ≈ 1,202,604
        차이가 7배 이상 벌어집니다.
        모델: "헐! 0.1 차이가 이렇게 큰 거였어? 정답을 확실하게 1.0에 가깝게 만들고, 오답은 -1로 쳐박아야겠구나!"
        즉, Temperature는 좁은 점수판(-1~1)을 넓게(예: -20~20) 늘려줘서 모델이 확실하게 구분하도록 강제하는 '확대경'입니다. """

        # 1. supervised contrastive sum over positives
        # “모든 positive를 합산해서 ratio 계산” → 카테고리별 embedding을 한 점에 모으는 경향
        loss = -torch.log(
            torch.exp(pos_sim).sum(dim=1) / 
            (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
        ).mean()
        # 2. LoRA + Classification
        # 3. loss = alpha * loss_contrastive + (1-alpha) * loss_cls
        # 1,2,3 방법 정확히 뭐가 다른건지를 모르겠네
        """
        1. Supervised Contrastive (현재 코드)	임베딩 구조 정렬 (Retrieval에 최적). 같은 카테고리끼리 뭉치게 만듭니다.
        2. LoRA + Classification	최종 분류 예측 (Classification에 최적). 임베딩 위에 분류 헤드를 붙여 분류 정확도를 높입니다.
        3. Hybrid Loss (가중 합)	두 마리 토끼 잡기 (가장 강력). 1번 Loss로 구조를 잡고, 2번 Loss로 카테고리 경계를 명확히 하여 임베딩 품질과 분류 정확도를 동시에 높입니다.
        현재는 1번 방식만으로도 목표 달성(임베딩 구조 정렬)에 충분합니다. 굳이 2, 3번으로 복잡하게 갈 필요는 없습니다.
        """
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_train_loss += loss.item()
        
    train_loss = total_train_loss / len(train_loader)
  
    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "valid/top1_acc": metrics["top1_acc"],
        "valid/top10_acc": metrics["topk_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

[Before Training...]
Top-1 Accuracy : 0.4551 | Top-10 Accuracy : 0.3496
Precision@10   : 0.3496 | MRR             : 0.5831


Epoch: 1: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [01:46<00:00,  4.79it/s]


[Epoch 1] Train Loss: 1.1509
Top-1 Accuracy : 0.4394 | Top-10 Accuracy : 0.3458
Precision@10   : 0.3458 | MRR             : 0.5695


Epoch: 2: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [01:47<00:00,  4.78it/s]


[Epoch 2] Train Loss: 1.0417
Top-1 Accuracy : 0.3982 | Top-10 Accuracy : 0.3211
Precision@10   : 0.3211 | MRR             : 0.5316


...

[Epoch 18] Train Loss: 1.0508
Top-1 Accuracy : 0.4904 | Top-10 Accuracy : 0.4285
Precision@10   : 0.4285 | MRR             : 0.6106


Epoch: 19: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [01:47<00:00,  4.78it/s]


[Epoch 19] Train Loss: 1.0452
Top-1 Accuracy : 0.4914 | Top-10 Accuracy : 0.4308
Precision@10   : 0.4308 | MRR             : 0.6116


Epoch: 20: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 512/512 [01:47<00:00,  4.78it/s]


[Epoch 20] Train Loss: 1.0570
Top-1 Accuracy : 0.4932 | Top-10 Accuracy : 0.4308
Precision@10   : 0.4308 | MRR             : 0.6117


In [18]:
# metrics = compute_retrieval_accuracy(model, valid_loader, device)
# wandb.log({
#     "epoch": 0,
#     "train_loss": 2,
#     "valid/top1_acc": metrics["top1_acc"],
#     "valid/top10_acc": metrics["topk_acc"],
#     "valid/precision@10": metrics["precision@k"],
#     "valid/mrr": metrics["mrr"],
#     "lr": LEARNING_RATE,
# })
# print(f"[Before Training...]")
# print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
# print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

# for epoch in range(20):
#     model.train()
#     total_train_loss = 0

#     for batch_inputs, labels in tqdm(train_loader, desc = f"Epoch: {epoch+1}"):
#         batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
#         labels = labels.to(device)
        
#         outputs = model(**batch_inputs)
#         embeddings = outputs.last_hidden_state.mean(dim=1)
#         embeddings = F.normalize(embeddings, p=2, dim=1)
#         similarity = torch.matmul(embeddings, embeddings.T)
        
#         labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
#         identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
#         labels_eq = labels_eq & (~identity_mask) # 자기 자신 제외

#         pos_mask = labels_eq.float()   # (N, N)

    
#         # [핵심 수정 1] "정답지(labels)"를 기준으로 유효 데이터 판별
#         # exp_pos 값(모델 예측값)을 보지 말고, mask(실제 정답 존재 여부)를 봐야 함!
#         valid_rows = pos_mask.sum(dim=1) > 0
        
#         if not valid_rows.any():
#             continue
        
#         """
#         pos_sim = similarity * pos_mask # (N, N) → anchor i의 positive 여러 개 (동일 label 여러 개)
#         여기서 pos_mask는 정답(Positive)이 아닌 곳을 0으로 만듭니다. 
#         문제는 유사도 점수(Logit)가 0이 된다고 해서, 확률(exp) 계산에서 사라지는 게 아니라는 점입니다.

#         "나랑 같은 카테고리가 아닌 놈들(Mask=0)"을 "유사도 0인 상태(exp=1)"로 만들어서 분자와 분모에 다 더해버리고 있습니다. 
#         유사도 0은 코사인 유사도에서 '직각'을 의미하며, 이는 꽤 의미 있는 관계일 수 있습니다(완전 반대인 -1보다 훨씬 큼). 
#         아예 무시(0으로 취급)하려면 exp 결과가 0이 되어야 합니다.
#         """
#         # 정답이 아닌 곳(0)에는 아주 작은 값(-1e9)을 넣어서 exp 계산 시 0이 되도록 해야 함
#         pos_sim = similarity.masked_fill(~pos_mask.bool(), -1e9)
#         neg_sim = hard_negative(embeddings, labels) # (N, k) → anchor i의 negative k개

#         temperature = 0.05 # 타우
#         pos_sim = pos_sim / temperature
#         neg_sim = neg_sim / temperature
#         """ 정답 점수: 0.8 / 0.05 = 16
#         오답 점수: 0.7 / 0.05 = 14
#         exp(16) ≈ 8,886,110 vs exp(14) ≈ 1,202,604
#         차이가 7배 이상 벌어집니다.
#         모델: "헐! 0.1 차이가 이렇게 큰 거였어? 정답을 확실하게 1.0에 가깝게 만들고, 오답은 -1로 쳐박아야겠구나!"
#         즉, Temperature는 좁은 점수판(-1~1)을 넓게(예: -20~20) 늘려줘서 모델이 확실하게 구분하도록 강제하는 '확대경'입니다. """

#         # 1. supervised contrastive sum over positives
#         # “모든 positive를 합산해서 ratio 계산” → 카테고리별 embedding을 한 점에 모으는 경향
#         # loss = -torch.log(
#         #     torch.exp(pos_sim).sum(dim=1) / 
#         #     (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
#         # ).mean()
#         # 2. LoRA + Classification
#         # 3. loss = alpha * loss_contrastive + (1-alpha) * loss_cls
#         # 1,2,3 방법 정확히 뭐가 다른건지를 모르겠네
#         """
#         1. Supervised Contrastive (현재 코드)	임베딩 구조 정렬 (Retrieval에 최적). 같은 카테고리끼리 뭉치게 만듭니다.
#         2. LoRA + Classification	최종 분류 예측 (Classification에 최적). 임베딩 위에 분류 헤드를 붙여 분류 정확도를 높입니다.
#         3. Hybrid Loss (가중 합)	두 마리 토끼 잡기 (가장 강력). 1번 Loss로 구조를 잡고, 2번 Loss로 카테고리 경계를 명확히 하여 임베딩 품질과 분류 정확도를 동시에 높입니다.
#         현재는 1번 방식만으로도 목표 달성(임베딩 구조 정렬)에 충분합니다. 굳이 2, 3번으로 복잡하게 갈 필요는 없습니다.
#         """
#         # --- [여기서부터 NaN 방지 로직] ---
        
#         # 분자: Positive들의 exp 합
#         exp_pos = torch.exp(pos_sim).sum(dim=1)
        
#         # 분모: 전체 합 (Positive + Negative)
#         # (Negative는 이미 hard_negative 함수 거쳐서 나온 거라 가정)
#         exp_neg = torch.exp(neg_sim).sum(dim=1)
#         exp_total = exp_pos + exp_neg + 10.0
        
#         # [핵심 수정 2] epsilon을 더해서 log(0)만 방지하되, 값은 그대로 살림
#         # 여기서 1e-6 대신 더 작은 1e-12를 써서 정밀도 유지
#         log_pos = torch.log(exp_pos + 1e-12)
#         log_total = torch.log(exp_total + 1e-12)
        
#         loss_per_anchor = -(log_pos - log_total)
        
#         # [최종] 정답이 실제 존재하는 행(valid_rows)만 평균
#         loss = loss_per_anchor[valid_rows].mean()
    
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         scheduler.step()
        
#         total_train_loss += loss.item()
        
#     train_loss = total_train_loss / len(train_loader)
  
#     model.eval()
#     metrics = compute_retrieval_accuracy(model, valid_loader, device)
#     wandb.log({
#         "epoch": epoch + 1,
#         "train_loss": train_loss,
#         "valid/top1_acc": metrics["top1_acc"],
#         "valid/top10_acc": metrics["topk_acc"],
#         "valid/precision@10": metrics["precision@k"],
#         "valid/mrr": metrics["mrr"],
#         "lr": scheduler.get_last_lr()[0],
#     })
#     print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
#     print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
#     print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

In [20]:
ㅋㅋ

NameError: name 'ᄏᄏ' is not defined

In [ ]:
# LR Range Test(Leslie Smith의 Cyclical LR 논문에서 나온 방식)

# --- 0. 하이퍼파라미터 및 설정 ---
# 이 값은 실험 중인 배치 사이즈와 K값, 그리고 Temperature입니다.
# Temperature는 반드시 적용해야 합니다! (이전 대화 참고)
temperature = 0.05
HARD_NEG_K = 3 
LR_START = 1e-7 # LR 탐색 시작점
GAMMA = 1.05    # LR 증가율 (스텝마다 5%씩 증가)
MAX_STEPS = 200 # 탐색 스텝 수
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. Hard Negative Mining 함수 (Vectorized) ---
def hard_negative_vectorized(embeddings, labels, k=HARD_NEG_K):
    similarity = torch.matmul(embeddings, embeddings.T)

    # 그래프 안정성을 위해 similarity 복사본 사용 (In-place 연산 때문)
    similarity_safe = similarity.clone()
    
    # Positive Mask 생성 (같은 라벨인 곳 True)
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)
    
    # 같은 라벨(Positive + 자기자신)은 유사도를 -무한대로 보내 topk에서 제외
    similarity_safe.masked_fill_(pos_mask, -1e9) 

    # 가장 높은 유사도를 가진 k개(Hard Negatives) 추출
    hard_neg_sims, _ = similarity_safe.topk(k, dim=1) 
    
    return hard_neg_sims

# --- 2. 모델 및 데이터 로드 (가정) ---
model_name = "intfloat/e5-small" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, lora_config)
model.to(device)

# ⚠️ 주의: 실제 데이터셋과 DataLoader는 정의되어 있다고 가정합니다.
# total_len = len(dataset)
# train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])
# train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
# -----------------------------------------------------------------------------------

## 🚀 Learning Rate 탐색을 위한 전체 코드 ##
print(f"--- 🚀 LR Range Test 시작: {LR_START:.8f}에서 시작 (스텝마다 {GAMMA}배 증가) ---")

optimizer = optim.AdamW(model.parameters(), lr=LR_START) # 아주 작은 LR로 시작
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=GAMMA) # LR을 지수적으로 증가

model.train()
logs = []

# train_loader 대신 tqdm을 사용하여 진행 상황을 시각화할 수 있습니다.
for i, (batch_inputs, labels) in tqdm(enumerate(train_loader), total=MAX_STEPS):
    batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
    labels = labels.to(device)
    
    # 1. 모델 순전파 (Forward Pass)
    outputs = model(**batch_inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    # 2. 유사도 및 Hard Negative 계산
    similarity = torch.matmul(embeddings, embeddings.T)
    
    # pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)
    labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)  # bool
    identity_mask = torch.eye(len(labels), device=device).bool()
    pos_mask = labels_eq & (~identity_mask) # 자기 자신 제외

    valid_rows = pos_mask.sum(dim=1) > 0
        
    if not valid_rows.any():
        continue
    
    # pos_sim = similarity * pos_mask.float()
    pos_sim = similarity.masked_fill(~pos_mask.bool(), -1e9)
    neg_sim = hard_negative_vectorized(embeddings, labels, k=HARD_NEG_K)
    
    # 3. Temperature Scaling 적용 (이전 대화 참고)
    # ⚠️ 이 부분이 없으면 Loss가 제대로 작동하지 않습니다!
    pos_sim = pos_sim / temperature
    neg_sim = neg_sim / temperature


    # 4. Loss 계산 (Multi-Positive InfoNCE Loss 변형)
    # torch.exp(pos_sim)의 sum(dim=1)을 통해 각 쿼리에 대한 모든 정답 유사도를 합산
    # loss = -torch.log(
    #     torch.exp(pos_sim).sum(dim=1) / 
    #     (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
    # )
    # loss = loss.mean()
    # 분자: Positive들의 exp 합
    exp_pos = torch.exp(pos_sim).sum(dim=1)
    exp_neg = torch.exp(neg_sim).sum(dim=1)
    exp_total = exp_pos + exp_neg
    
    # [핵심 수정 2] epsilon을 더해서 log(0)만 방지하되, 값은 그대로 살림
    # 여기서 1e-6 대신 더 작은 1e-12를 써서 정밀도 유지
    log_pos = torch.log(exp_pos + 1e-12)
    log_total = torch.log(exp_total + 1e-12)
    
    loss_per_anchor = -(log_pos - log_total)
    
    # [최종] 정답이 실제 존재하는 행(valid_rows)만 평균
    loss = loss_per_anchor[valid_rows].mean()
    
    # 5. 역전파 및 최적화 (Backward Pass)
    optimizer.zero_grad()
    loss.backward()
    
    # ⚠️ RuntimeError 방지 팁: Loss 값이 너무 커지면 Gradient가 폭발하여 Nan이 발생할 수 있습니다.
    if torch.isnan(loss) or loss.item() > 100:
        print(f"\nLoss 폭발 (Step {i}): {loss.item():.4f}. 탐색 중단.")
        break
        
    optimizer.step()
    scheduler.step() # LR 키우기

    # 6. 로그 기록
    current_lr = optimizer.param_groups[0]['lr']
    logs.append({'step': i, 'lr': current_lr, 'loss': loss.item()})
    
    # 현재 LR과 Loss 출력
    if i % 10 == 0: # 너무 자주 출력되지 않도록 10 스텝마다 출력
        print(f"Step {i}, LR: {current_lr:.8f}, Loss: {loss.item():.4f}")

    if i >= MAX_STEPS:
        break

print("--- 🔍 LR Range Test 완료 ---")

In [ ]:
import matplotlib.pyplot as plt

plt.plot([l['lr'] for l in logs], [l['loss'] for l in logs])
plt.xscale('log')
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.show()

In [ ]:
    import pandas as pd
    from transformers import AutoTokenizer
    import matplotlib.pyplot as plt
    
    # 사용 중인 모델의 토크나이저 로드
    tokenizer = AutoTokenizer.from_pretrained("intfloat/e5-small")
    
    # 가정: df['combined_text']에 "제목 + 줄거리"가 들어있다고 칩시다.
    # 토큰 개수 세기
    books['token_count'] = books['text'].apply(lambda x: len(tokenizer.encode(x)))
    
    # 통계 요약 출력
    print(books['token_count'].describe())
    
    # 시각화 (히스토그램)
    plt.hist(books['token_count'], bins=50)
    plt.title("Book Token Length Distribution")
    plt.xlabel("Token Count")
    plt.ylabel("Frequency")
    plt.show()